# LeNet-5: Modern Apache TVM Relax + MetaSchedule on NVIDIA GPUs

**Project:** [Source code](https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization/tree/main/projects/lenet5-relax-metaschedule) · **Author:** Fardeen Shroff

**Hypothesis:** Given the same LeNet-5 trained weights, input shape, FP32 dtype, CUDA device, and timing protocol, can MetaSchedule-tuned TensorIR improve inference versus Relax's default GPU schedules?

**Critical distinction:** The previous study's 18.03× measurement compared PyTorch CPU to GPU. This notebook measures *untuned Relax versus MetaSchedule-tuned Relax on one NVIDIA GPU*. It does not automatically reproduce the earlier AutoTVM paper.

**First:** Runtime → Change runtime type → **T4 GPU**, then reconnect. Free GPU availability and session limits are subject to Google Colab.


In [6]:
import sys, os, json, subprocess, pathlib, shutil, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No NVIDIA GPU runtime. Set Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
subprocess.run(['nvidia-smi'], check=True)


Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)


CompletedProcess(args=['nvidia-smi'], returncode=0)

## Install the modern TVM compiler

Apache TVM 0.27.0.post1 uses `Relax`, `MetaSchedule`, and bundled TIRx (`tvm.tirx`); this is different from old Relay/AutoTVM. If the wheel fails to initialize CUDA, refer to the source-build instructions in this repository rather than posting invalid benchmark numbers.


In [7]:
repo = pathlib.Path('/content/lenet5-gpu-autotvm-optimization')
url = 'https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization.git'
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', url, str(repo)], check=True)
project = repo / 'projects' / 'lenet5-relax-metaschedule'
assert (project / 'pyproject.toml').is_file(), f'Project not found: {project}'
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'apache-tvm[cuda,torch,meta-schedule]==0.27.0.post1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'apache-tvm==0.27.0.post1'], check=True)
print('Project path:', project)


Project path: /content/lenet5-gpu-autotvm-optimization/projects/lenet5-relax-metaschedule


## Preflight: verify TVM runtime and CPU-safe project tests

A GPU is considered **available** only when both PyTorch and the TVM CUDA runtime can access it. This is a prerequisite check, not a claim that MetaSchedule has improved latency.


In [8]:
subprocess.run([sys.executable, '-m', 'relax_meta.environment'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## Train once for five epochs, or provide an existing compatible checkpoint

The compiled baselines use identical learned weights. Training here is a *checkpoint preparation step* and is not MetaSchedule tuning. If `checkpoints/lenet5_5epochs.pt` already exists in this Colab runtime, this cell reuses it.


In [9]:
checkpoint = pathlib.Path('checkpoints/lenet5_5epochs.pt')
if not checkpoint.exists():
    subprocess.run([sys.executable, '-m', 'relax_meta.train',
                    '--epochs', '5', '--device', 'cuda', '--batch-size', '256',
                    '--checkpoint', str(checkpoint),
                    '--output', 'results/training.json'], check=True)
else:
    print('Reusing existing trained checkpoint:', checkpoint)


Reusing existing trained checkpoint: checkpoints/lenet5_5epochs.pt


## Small MetaSchedule smoke experiment

Run batch size 1 with **16 total search trials** before committing more free GPU time. This experiment must write a tuning database **and** pass numerical comparison against PyTorch. If compilation or import is incompatible with the Colab TVM build, the notebook stops with a real error (no synthetic speedup values).


In [ ]:
subprocess.run([sys.executable, '-m', 'relax_meta.benchmark',
                '--checkpoint', str(checkpoint), '--batch-sizes', '1',
                '--trials', '16', '--max-trials-per-task', '8',
                '--warmup', '10', '--repeats', '3', '--number', '20',
                '--output', 'results/meta_smoke_batch1.json'], check=True)
print(json.dumps(json.loads(pathlib.Path('results/meta_smoke_batch1.json').read_text())['batches'][0]['speedups'], indent=2))


## Full experiment — batch sizes 1 and 128

Tune each fixed-shape model independently, using 128 total search trials (default) per shape. A previously recorded batch-1 database may be warm-started. The primary metric is *untuned Relax median latency / tuned Relax median latency*. Report values **below 1.0×** as tuning regressions, not improvements.

This is computationally expensive compared with PyTorch inference. The free GPU session may expire; adjust trial counts downward if needed, but always record the reduced budget in your results.


In [ ]:
subprocess.run([sys.executable, '-m', 'relax_meta.benchmark',
                '--checkpoint', str(checkpoint), '--batch-sizes', '1', '128',
                '--trials', '128', '--max-trials-per-task', '16',
                '--warmup', '20', '--repeats', '7', '--number', '50',
                '--output', 'results/relax_metaschedule_comparison.json'], check=True)
report = json.loads(pathlib.Path('results/relax_metaschedule_comparison.json').read_text())
for batch in report['batches']:
    print('Batch', batch['batch_size'])
    print('Schedule records:', batch['tuning']['records_after'])
    print('Speedups:', json.dumps(batch['speedups'], indent=2))


## Save experiment evidence

Download the JSON reports, training checkpoint, and complete schedule database. These are **measured outputs** produced by this run. Upload them here for verification and a GitHub results table. The archive is created only after the benchmark reaches this cell successfully.


In [ ]:
import zipfile
archive = pathlib.Path('/content/lenet5_relax_metaschedule_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('results', 'tuning_logs', 'checkpoints'):
        for f in pathlib.Path(folder).rglob('*'):
            if f.is_file():
                z.write(f, arcname=str(f))
print('Archive:', archive, archive.stat().st_size, 'bytes')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Download the file from:', archive)
